# Ler base de dados

In [1]:
import pandas as pd
import numpy as np
equipDB = pd.read_csv("EquipDB.csv", header=None, names=["ID_equipamento", "TempoAposFalha", "Cluster", "CustoDeFalha"])
ClusterDB= pd.read_csv("ClusterDB.csv", header=None, names=["ID_Cluster", "eta", "beta"])
MPDB=pd.read_csv("MPDB.csv", header=None, names=["ID_plano_risco", "Fator_risco(k)", "CustoDoPlano"])

# Unir informações do ClusterDB ao equipDB
equipDB = equipDB.merge(ClusterDB, left_on="Cluster", right_on="ID_Cluster")


# Formula de Weibull

$P_{ij}$ é a probabilidade de falha do equipmento $i$ sob o plano de manutenção $j$

In [2]:
def Fi(t, eta, beta):
    return 1-np.exp(-(t/eta)**beta)

def P(t0, eta, beta, k, delta_t):
    return (Fi((t0+k*delta_t), eta=eta, beta=beta)-Fi(t=t0, eta=eta, beta=beta))/(1-Fi(t=t0, eta=eta, beta=beta))

# Modelo Matematicos

## Modelo para $f2(*)$


## Conjuntos e Parâmetros
$I$: Conjunto de equipamentos ($|I| = 500$, $i \in I$)

$J = \{1,2,3\}$: Planos de manutenção disponíveis ($j \in J$)

$C = \{1,2,3,4\}$: Clusters de equipamentos ($c \in C$)

$t_0^i$: Idade atual do equipamento $i$ (em anos)

$\text{c}_i$: Cluster ao qual pertence o equipamento $i$

$\text{f}_i$: Custo associado à falha do equipamento $i$

$k_j$: Fator de risco do plano de manutenção $j$

$\text{m}_j$: Custo de aplicar o plano $j$ a um equipamento

$\eta_c$: Parâmetro de escala (Weibull) para o cluster $c$

$\beta_c$: Parâmetro de forma (Weibull) para o cluster $c$

$p_{i,j}$: probabilidade de falha do equipmento $i$ sob o plano de manutenção $j$

$\Delta t = 5$ anos: Horizonte de planejamento


### Varaivel de decisão
$$
x_{i,j} = 
\begin{cases} 
1, & \text{se o equipamento } i \text{ for alocado ao plano } j \\
0, & \text{caso contrário}
\end{cases} $$

$$
\begin{align}
 f_1(\mathbf{x}) &=\min \sum_{i \in I} \sum_{j \in J} \text{m}_j \cdot x_{i,j} \quad \text{(Custo total de manutenção)} \\
f_2(\mathbf{x}) &= \min \sum_{i \in I} \sum_{j \in J} p_{i,j} \cdot \text{f}_i \cdot x_{i,j} \quad \text{(Custo esperado de falha)}
\end{align}$$
onde: 

$$
\begin{align}
p_{i,j} = \frac{F_{c(i)}(t_0^i + k_j \Delta t) - F_{c(i)}(t_0^i)}{1 - F_{c(i)}(t_0^i)} \\
F_c(t) = 1 - \exp\left[-\left(\frac{t}{\eta_c}\right)^{\beta_c}\right]
\end{align}$$

## Restrições

Alocação obrigatória
$$
\begin{equation}
\sum_{j \in J} x_{i,j} = 1 \quad \forall i \in I
\end{equation}
$$

Integralidade:

$$
\begin{equation}
x_{i,j} \in \{0,1\} \quad \forall i \in I, \forall j \in J
\end{equation}
$$

## Variavel de decisão

In [3]:
# Criar a matriz  de probabilidade Pij
Pij = np.zeros((len(equipDB), len(MPDB)))
delta_t = 5  # Definir um valor para Delta_t,

for i, equipamento in equipDB.iterrows():
    t0 = equipamento["TempoAposFalha"]
    eta = equipamento["eta"]
    beta = equipamento["beta"]
    
    for j, plano in MPDB.iterrows():
        k = plano["Fator_risco(k)"]
        Pij[i, j] = P(t0, eta, beta, k, delta_t)

# Converter a matriz para um DataFrame para melhor visualização
Pij_df = pd.DataFrame(Pij, index=equipDB["ID_equipamento"], columns=MPDB["ID_plano_risco"])


## Geração Solução Inicial (Cálculo do Índice Crítico)
Para cada equipamento $i$, calcule o índice de criticidade:

$$
\text{Índice}_i = \text{custo\_falha}_i \times p_{i,\text{nenhuma}} \times t_0^i
$$

onde:

$\text{custo\_falha}_i$: Custo associado à falha do equipamento $i$

$p_{i,\text{nenhuma}}$: Probabilidade de falha se nenhuma manutenção for aplicada (plano $j=1$)

$t_0^i$: Tempo de operação

## Alocação de Planos de Manutenção

Ordene os equipamentos em ordem $\textbf{decrescente}$ de $\text{Índice}_i$ e aloque os planos:


$\textbf{Top 20\%}$ equipamentos: Manutenção detalhada (plano $j=3$)


$\textbf{Próximos 30\%}$: Manutenção intermediária (plano $j=2$)

$\textbf{Restante 50\%}$: Nenhuma manutenção (plano $j=1$)

In [4]:
def gerar_Xij_inicial(equipDB, MPDB, Pij):
    """Gera uma solução inicial balanceada para Xij."""
    n_equip = len(equipDB)
    Xij = np.zeros((n_equip, len(MPDB)))
    
    # Critério de criticidade: custo_falha * probabilidade sem manutenção
    criticidade = equipDB["CustoDeFalha"].values * Pij[:, 0]*equipDB["TempoAposFalha"]
    equip_ordenados = np.argsort(-criticidade)  # Ordem decrescente
    
    # Distribuição 20% detalhada, 30% intermediária, 50% nenhuma
    for idx, i in enumerate(equip_ordenados):
        if idx < 0.2 * n_equip:
            Xij[i, 2] = 1  # Plano 3 (detalhada)
        elif idx < 0.5 * n_equip:
            Xij[i, 1] = 1  # Plano 2 (intermediária)
        else:
            Xij[i, 0] = 1  # Plano 1 (nenhuma)
    
    return Xij
Xij=gerar_Xij_inicial(equipDB=equipDB, MPDB=MPDB, Pij=Pij)

### Calcula o fit do ploblema

In [5]:
mj=MPDB.CustoDoPlano.values
fi=equipDB.CustoDeFalha.values
def fit_f1(X, m):
    """
    Calcula o custo total de manutenção.

    Parâmetros:
    - x: matriz numpy (n x 3) representando as variáveis de decisão x_{i,j}
    - m: vetor numpy (3,) representando os custos de manutenção m_j para cada coluna

    Retorna:
    - Custo total de manutenção (float)
    """
    return np.sum(X*m)

def fit_f2(X, p, f):
    """
    Calcula o custo esperado de falha.

    Parâmetros:
    - x: matriz numpy (n x 3) representando as variáveis de decisão x_{i,j}
    - p: matriz numpy (n x 3) representando as probabilidades de falha p_{i,j}
    - f: vetor numpy (n,) representando os custos de falha f_i

    Retorna:
    - Custo esperado de falha (float)
    """
    return np.sum(p * f[:, np.newaxis] * X)
print("Custo total de manutenção:",fit_f1(Xij,mj))
print("Custo esperado de falha:", fit_f2(Xij,Pij,fi))

Custo total de manutenção: 350.0
Custo esperado de falha: 1461.1009514401871


## Gerar tres estruturas de visinhaça


### Vizinhança 1 - Move

Move um elemento do plano 𝑗  para o plano l.

Objetivo → Explorar novas combinaçõe

### Vizinhaça 2 - one-opt
Escolhe 1 par de elementos de 2 planos diferentes e troca suas posições.

Objetivo → Perturbação a solução

### Vizinhaça 3 - Cycle Shift entre Planos
Rotaciona 3 elementos entre 3 planos diferentes, sendo que pelos menos 2 elementos são diferenres:

elemento do plano A → vai pro plano B

B → vai pro C

C → vai pro A

Objetivo → Movimentação circular, gera boa perturbação.

In [6]:
import random

def move(X):
    i = random.randint(0, len(X)-1)
    plano_atual = int(X[i].argmax())
    novo_plano = random.choice([p for p in range(len(X[0])) if p != plano_atual])
    X[i][plano_atual] = 0
    X[i][novo_plano] = 1

# Vizinhança 2 - 2-opt entre planos
def one_opt(X):
    for i in range(10):
        par = random.sample(range(len(X)), 2)
        if not np.array_equal(X[par[0]], X[par[1]]):
            X[par[0]], X[par[1]]= X[par[1]].copy(), X[par[0]].copy()
            break

def cycle_shift(X):
    for _ in range(10):
        i1, i2, i3 = random.sample(range(len(X)), 3)
        if len({tuple(X[i1]), tuple(X[i2]), tuple(X[i3])}) > 1:
            break

    p1 = int(X[i1].argmax())
    p2 = int(X[i2].argmax())
    p3 = int(X[i3].argmax())

    print(X[i1],X[i2],X[i3])

    X[i1][p1], X[i1][p2] = 0, 1
    X[i2][p2], X[i2][p3] = 0, 1
    X[i3][p3], X[i3][p1] = 0, 1

In [7]:

def vizinhanca_1(solucao):
    nova = solucao.copy()
    i = np.random.randint(len(solucao))
    nova[i] = np.random.choice([p for p in [1, 2, 3] if p != solucao[i]])
    return nova

def vizinhanca_2(solucao):
    nova = solucao.copy()
    i, j = np.random.choice(len(solucao), 2, replace=False)
    for idx in [i, j]:
        nova[idx] = np.random.choice([p for p in [1, 2, 3] if p != solucao[idx]])
    return nova

def vizinhanca_3(solucao):
    nova = solucao.copy()
    i = np.random.randint(0, len(solucao) - 10)
    for j in range(i, i + 10):
        nova[j] = np.random.choice([p for p in [1, 2, 3] if p != solucao[j]])
    return nova

vizinhas = [vizinhanca_1, vizinhanca_2, vizinhanca_3]


In [8]:

def gerar_solucao_inicial(equipDB, matriz_pij):
    criticidade = equipDB["custo_falha"].values * matriz_pij[:, 0]
    ordem = np.argsort(-criticidade)
    solucao = np.ones(len(equipDB), dtype=int)
    n = len(equipDB)
    for idx, i in enumerate(ordem):
        if idx < 0.2 * n:
            solucao[i] = 3
        elif idx < 0.5 * n:
            solucao[i] = 2
        else:
            solucao[i] = 1
    return solucao


In [9]:

def busca_local(solucao, obj_func, *args):
    melhor = solucao.copy()
    melhor_valor = obj_func(melhor, *args)
    for viz in vizinhas:
        candidato = viz(melhor)
        valor = obj_func(candidato, *args)
        if valor < melhor_valor:
            return candidato
    return melhor
